# Mercor Cheating Detection: Cost-Aware Classification Pipeline

**Team Members:** Saurav Rijal, Shivendra Bhagat

### Problem
Binary classification to detect cheating during online AI-driven interviews. The objective is to **minimize total economic cost**, where missing a cheater costs $600, incorrectly blocking a legitimate candidate costs $300, and false flags in the review queue cost $150.

### Approach
1. **Preprocessing:** Median imputation, outlier detection, log transforms, MI-guided interaction terms, and social graph features (degree centrality, neighbourhood cheating density, Louvain community structure).
2. **Model Progression:** Logistic Regression → Decision Tree → Random Forest → XGBoost → LightGBM → Ensemble blend.
3. **Calibration:** Per-fold Platt/Isotonic calibration to fix tree overconfidence.
4. **Evaluation:** Custom cost function with threshold sweep across Auto-Pass / Review / Auto-Block zones.

In [ ]:
import os, glob, shutil

# Auto-detect Kaggle dataset paths
data_dirs = glob.glob('/kaggle/input/**/train.csv', recursive=True)
code_dirs = glob.glob('/kaggle/input/**/train.py', recursive=True)

DATA_DIR = os.path.dirname(data_dirs[0]) if data_dirs else '/kaggle/input/mercor-cheating-detection/'
WORK_DIR = '/kaggle/working/cheating'
os.makedirs(f'{WORK_DIR}/data', exist_ok=True)

# Copy codebase
if code_dirs:
    CODE_ROOT = os.path.dirname(code_dirs[0])
    print(f'Found codebase at: {CODE_ROOT}')
    os.system(f'cp {CODE_ROOT}/*.py {WORK_DIR}/')
    os.system(f'cp {CODE_ROOT}/requirements.txt {WORK_DIR}/ 2>/dev/null || true')

# Link competition data
for f in ['train.csv', 'test.csv', 'social_graph.csv', 'feature_metadata.json']:
    src, dst = os.path.join(DATA_DIR, f), os.path.join(WORK_DIR, 'data', f)
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)
        print(f'Linked {f}')

In [ ]:
!cd {WORK_DIR} && pip install -r requirements.txt -q 2>/dev/null || true

### Execute Full Pipeline
Trains all 5 models, calibrates, blends, optimizes thresholds, and generates `submission.csv`.

In [ ]:
!cd {WORK_DIR} && python3 train.py --no-pseudo --data-dir data

In [ ]:
import shutil, os
WORK_DIR = '/kaggle/working/cheating'
src = f'{WORK_DIR}/outputs/submissions/submission.csv'
if os.path.exists(src):
    shutil.copy(src, '/kaggle/working/submission.csv')
    print('Submission ready at /kaggle/working/submission.csv')
else:
    print('ERROR: submission.csv not found')